## 02.2 — Users Data Preprocessing


In [1]:
import os
import shutil
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,concat_ws, count
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

## THE NEXT TWO CELLS ARE CRUCIAL FOR RUNNING HADOOP AND PYSPARK ON MY DEVICE


In [2]:
import sys

# On Windows, the system-level SPARK_HOME may point to a different Spark version.
# We override it here to make sure PySpark always finds its own bundled JARs.
os.environ["SPARK_HOME"]  = os.path.dirname(pyspark.__file__)

# Spark needs Hadoop's winutils.exe on Windows to handle local file operations
# (directory creation, temp files, etc.). We installed winutils at C:\hadoop\bin\.
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Prepend C:\hadoop\bin to PATH so the JVM can find hadoop.dll via
# System.loadLibrary("hadoop"). HADOOP_HOME alone only locates winutils.exe
# for Hadoop's shell helpers — it does NOT add the directory to
# java.library.path. Without this, NativeIO$Windows.access0 throws
# UnsatisfiedLinkError on Spark write operations.
hadoop_bin = r"C:\hadoop\bin"
if hadoop_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")

# Pin the Python interpreter for both the driver and the executor-side workers
# to the exact Python running this notebook. Without these, PySpark on Windows
# can pick a different Python for workers, causing them to die with
# "Connection reset" the moment Spark tries to deserialize Python rows.
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("SPARK_HOME    :", os.environ["SPARK_HOME"])
print("HADOOP_HOME   :", os.environ["HADOOP_HOME"])
print("PATH[0]       :", os.environ["PATH"].split(os.pathsep)[0])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])

SPARK_HOME    : c:\Users\Mariam\anaconda3\envs\spotify-recommender\Lib\site-packages\pyspark
HADOOP_HOME   : C:\hadoop
PATH[0]       : C:\hadoop\bin
PYSPARK_PYTHON: c:\Users\Mariam\anaconda3\envs\spotify-recommender\python.exe


In [ ]:
# local[*]  — uses all CPU cores on this machine (pseudo-distributed mode).
# spark.driver.memory — heap given to the JVM driver process.
# spark.local.dir     — MUST be on the same drive as the output directory (D:).
#   Spark writes temp files here, then renames them to the final output path.
#   Java's File.renameTo() cannot rename across different drives on Windows,
#   so if temp is on C: and output is on D:, every write fails silently.
# spark.hadoop.home.dir — tells the JVM where winutils lives at runtime.
# RawLocalFileSystem  — skips Unix chmod/chown calls that crash on Windows.
# fileoutputcommitter.algorithm.version=2 — avoids the rename-based commit phase.
# marksuccessfuljobs=false — skips writing the _SUCCESS marker file.

SPARK_TEMP_DIR = "D:/tmp/spark"
os.makedirs(SPARK_TEMP_DIR, exist_ok=True)

spark = SparkSession.builder \
    .appName("SpotifyPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.local.dir", SPARK_TEMP_DIR) \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


def spark_write_csv(df, relative_path):
    """
    Write a Spark DataFrame to a single CSV file on Windows.
    Deletes the output directory with Python first so Spark never has to
    rename or overwrite an existing folder across drives.
    """
    if os.path.exists(relative_path):
        shutil.rmtree(relative_path)
    abs_path = os.path.abspath(relative_path).replace("\\", "/")
    df.coalesce(1).write.csv(f"file:///{abs_path}", header=True)
    print(f"Saved → {relative_path}/")

In [ ]:
PROCESSED_DIR = "../data/processed"
SPLIT_DIR     = "../outputs/train_test_split"
RAW_DIR       = "../data/raw"

In [ ]:
df_als_raw = spark.read.csv(
    f"{RAW_DIR}/spotify_playlists/spotify_dataset.csv",
    header=True,
    quote='"',
    ignoreLeadingWhiteSpace=True,
    mode="DROPMALFORMED"
)

# The raw CSV has column names with leading spaces and quote characters,
# e.g. ' "user_id"'. We strip them to get clean names: user_id, trackname, etc.
clean_cols = [c.strip().strip('"') for c in df_als_raw.columns]
df_als = df_als_raw.toDF(*clean_cols)

# Drop rows missing user, track, or artist. We require artistname here because
# track_key = concat_ws("||", trackname, artistname) silently skips nulls,
# so a null artist would collapse different artists' same-titled tracks into
# one fake "song" and corrupt the ALS item space.
df_als = df_als.dropna(subset=["user_id", "trackname", "artistname"])

# Count how many times each user played each song (i.e. how many playlists
# the (user, track, artist) triple appears in). This is our implicit feedback
# signal: more appearances = stronger preference.
df_als = (
    df_als
    .groupBy("user_id", "trackname", "artistname")
    .agg(count("*").alias("play_count"))
)

# Cache so the two distinct() calls below don't re-read + re-aggregate the CSV.
df_als = df_als.cache()

# Combine track name + artist into one key to uniquely identify a song
# across different playlists (same song can have slightly different IDs).
df_als = df_als.withColumn("track_key", concat_ws("||", col("trackname"), col("artistname")))

# Encode user_id and track_key as sequential integers (0, 1, 2, ...).
# ALS in Spark MLlib requires integer IDs, not strings.
#
# We use a Window + row_number on the distinct values instead of:
#   - StringIndexer: collects all unique values to the driver → OOM at 2.7M songs
#   - rdd.zipWithIndex().map(...).toDF(...): pipes every row through a Python
#     worker (default 512 MB heap), which dies with "Connection reset by peer"
#     on Windows under load.
# row_number().over(Window.orderBy(...)) stays entirely in the JVM. Spark warns
# that all distinct values get pulled into a single partition for the global
# ordering, but that is bounded by the number of unique users / songs and fits
# comfortably in the 4 GB driver heap.
user_mapping = df_als.select("user_id").distinct() \
    .withColumn("user_id_enc", row_number().over(Window.orderBy("user_id")) - 1) \
    .withColumnRenamed("user_id", "user_id_original")

song_mapping = df_als.select("track_key").distinct() \
    .withColumn("song_id_enc", row_number().over(Window.orderBy("track_key")) - 1) \
    .withColumnRenamed("track_key", "track_key_ref")

# Join the integer encodings back to the main DataFrame
df_als = df_als.join(user_mapping, df_als["user_id"] == user_mapping["user_id_original"]) \
               .drop("user_id_original")
df_als = df_als.join(song_mapping, df_als["track_key"] == song_mapping["track_key_ref"]) \
               .drop("track_key_ref")

# Use play_count as the interaction strength — how many times the user played the song.
df_als = df_als.withColumn("interaction", col("play_count").cast("double")).drop("play_count")

# Keep only the columns needed for ALS training + human-readable labels for debugging.
# playlist_name is dropped — it's not meaningful after aggregation.
users_ds_final = df_als.select(
    col("user_id_enc").alias("user_id"),
    col("song_id_enc").alias("song_id"),
    col("interaction"),
    col("user_id").alias("user_id_original"),
    col("trackname").alias("track_name"),
    col("artistname").alias("artist_name")
)

print(f"Interactions : {users_ds_final.count():,}")
print(f"Columns : {users_ds_final.columns}")
print("Sample interactions:")
users_ds_final.show(5)
spark_write_csv(users_ds_final, f"{PROCESSED_DIR}/interactions_cleaned")
unique_users = users_ds_final.select("user_id").distinct() 
print(f"Unique users : {unique_users.count():,}")
print("Sample unique users:")
unique_users.show(5)
spark_write_csv(unique_users, f"{PROCESSED_DIR}/unique_users")

unique_songs = users_ds_final.select("song_id").distinct()
print(f"Unique songs : {unique_songs.count():,}")
print("Sample unique songs:")
unique_songs.show(5)
spark_write_csv(unique_songs, f"{PROCESSED_DIR}/unique_songs")


In [ ]:
# Split interactions 80/20.
users_train_ds, rest = users_ds_final.randomSplit([0.8, 0.2], seed=42)

users_test_ds, users_validation_ds = rest.randomSplit([0.5, 0.5], seed=42)
spark_write_csv(users_train_ds, f"{SPLIT_DIR}/users_train")
spark_write_csv(users_test_ds,  f"{SPLIT_DIR}/users_test")
spark_write_csv(users_validation_ds, f"{SPLIT_DIR}/users_validation")

print(f"ALS Train : {users_train_ds.count():,} rows")
print(f"ALS Test  : {users_test_ds.count():,} rows")
print(f"ALS Validation : {users_validation_ds.count():,} rows")

Saved → ../outputs/train_test_split/users_train/
Saved → ../outputs/train_test_split/users_test/
ALS Train : 9,107,237 rows
ALS Test  : 2,276,613 rows


In [ ]:

spark.stop()
print("Done. Spark session closed.")

Done. Spark session closed.
